In [37]:
import seaborn as sns
import plotly.express as px
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import umap
import plotly.io as pio
import os

from hbn.constants import Defaults
from hbn.visualization import visualize as vis

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

# pio.renderers.default = 'iframe'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
## Functions

def piechart(dataframe, y='PreInt_DevHx,birthweight_lbs', hue='DX_01'):
    
    if hue:
        df_grouped = dataframe.groupby(hue).agg({y: 'mean'}).reset_index()
        fig = px.pie(df_grouped, names=hue, values=y)
    else:
        fig = px.pie(dataframe, names=y)
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.show()

In [30]:
from hbn.data import make_dataset

# get participants
participants = pd.read_csv(os.path.join(Defaults.PHENO_DIR, 'participants.csv'))

# get summary of clinical diagnosis + other demographics
df_all = make_dataset.make_summary(save=False)
df_all['Age'] = df_all['Age'].round()
df_CGAS = make_dataset.add_CGAS_Score(df_all)

# get participants
df_part = pd.DataFrame(participants, columns=['Identifiers'])

# filter large dataframe to include only ADHD + No Diagnosis
df = df_all.merge(df_part, on='Identifiers')
df_CGAS = df_CGAS.merge(df_part, on='Identifiers')

df.head(5)

,Identifiers,Age,Sex,Enroll_Year,Administration,DX_01,DX_01_ByHx,DX_01_Cat,DX_01_Code,DX_01_Confirmed,...,"PreInt_Demos_Fam,Season","PreInt_Demos_Fam,Sep_Yrs","PreInt_Demos_Fam,Site","PreInt_Demos_Fam,SocialService","PreInt_Demos_Fam,Study","PreInt_Demos_Fam,Visit_label","PreInt_Demos_Fam,Year","PreInt_Demos_Fam,guardian_maritalstatus","PreInt_Demos_Fam,Child_Race_cat","PreInt_Demos_Fam,Child_Ethnicity_cat"
0,NDARAA075AMK,7.0,female,2016.0,All,No Diagnosis Given,0.0,No Diagnosis Given,No Diagnosis Given,NaN,...,Summer,NaN,2.0,NaN,HBN,NaN,2016.0,1.0,Unknown,Unknown
1,NDARAA112DMH,6.0,male,2016.0,All,ADHD-Combined Type,0.0,Neurodevelopmental Disorders,F90.2,NaN,...,Fall,NaN,1.0,NaN,HBN,NaN,2016.0,1.0,Hispanic,Hispanic or Latino
2,NDARAA117NEJ,7.0,male,2016.0,All,ADHD-Combined Type,0.0,Neurodevelopmental Disorders,F90.2,NaN,...,Winter,NaN,1.0,NaN,HBN,NaN,2017.0,1.0,Hispanic,Hispanic or Latino
3,NDARAA306NT2,21.0,female,2019.0,All,Generalized Anxiety Disorder,0.0,Anxiety Disorders,NaN,1.0,...,Summer,NaN,3.0,NaN,HBN,NaN,2019.0,1.0,Unknown,White/Caucasian
4,NDARAA358BPN,12.0,male,2017.0,All,No Diagnosis Given: Incomplete Eval,NaN,No Diagnosis Given: Incomplete Eval,No Diagnosis Given: Incomplete Eval,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Unknown


In [32]:
## summary stats

sample_size = len(df['Identifiers'].unique())
print(f'1. total sample size is {sample_size}\n')

sex = df['Sex'].value_counts()
print(f'2. there are {sex.male} males and {sex.female} females\n')

ages_6_10 = len(df[df['Age'].round()<=10])
ages_10_21 = len(df[df['Age'].round()>10])
print(f'3. there are {ages_6_10} children ages 6-10 and {ages_10_21} children ages 11-21\n')

num_sites = len(df['Site'].unique())
print(f'4. there are {num_sites} study sites\n')

years = df['Enroll_Year'].value_counts().index.astype(str).str.strip('.0').astype(int).tolist()
num_years = len(df['Enroll_Year'].unique())
print(f'5. data were collected across {num_years} years: {years}\n')

num_disorders = len(df['DX_01'].unique())
num_cat = len(df['DX_01_Cat'].unique())
print(f'6. there are {num_disorders} unique disorders, classified under {num_cat} categories\n')

comorbid = round((df['comorbidities'].value_counts() / len(df)) * 100)
num_comorbid = comorbid[1:].sum()
print(f'7. approximately {num_comorbid}% have disorder combordities\n')

disorder_cat = round((df['DX_01'].value_counts() / len(df)) * 100).head(1)
print(f'8. most prevalent diagnosis is {disorder_cat.index[0]} - {disorder_cat.values[0]}% of sample\n')

disorder = round((df['DX_01_Cat_new'].value_counts() / len(df)) * 100).head(1)
print(f'9. most prevalent category of diagnosis is {disorder.index[0]} - {disorder.values[0]}% of sample\n')

sex = df_CGAS.groupby(['Sex']).agg({'CGAS_Score': 'mean'})
f_cgas = sex.loc['female'].values[0]
m_cgas = sex.loc['male'].values[0]
print(f'10. females have an average general functioning score (CGAS) of {round(f_cgas)}% and males {round(m_cgas)}%\n')



1. total sample size is 4767

2. there are 3034 males and 1733 females

3. there are 2800 children ages 6-10 and 1967 children ages 11-21

4. there are 5 study sites

5. data were collected across 8 years: [2018, 2019, 2017, 202, 2016, 2021, 2015, 2022]

6. there are 72 unique disorders, classified under 17 categories

7. approximately 62.0% have disorder combordities

8. most prevalent diagnosis is ADHD-Combined Type - 18.0% of sample

9. most prevalent category of diagnosis is ADHD - 40.0% of sample

10. females have an average general functioning score (CGAS) of 66% and males 64%



## Majority of sample (n=4106) is male (n=2626) and majority of sample are in the age range: 6-10 (n=2394)

In [43]:
fig = px.histogram(df, x="Age", nbins=10, color="Sex")
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()

## ADHD make up the majority of diagnoses

In [97]:
piechart(dataframe=df, y='DX_01_Cat_new', hue=None)


## Majority of samples were collected in study site 1

In [45]:
piechart(dataframe=df, y='Site', hue=None)

## Majority of participans were collected between 2017-2019

In [46]:
piechart(dataframe=df, y='Enroll_Year', hue=None)

In [99]:
#vis.wordcloud(dataframe=df, column='DX_01')

## ADHD is the most diagnosed disorder in the HBN followed by ASD. Most disorders are diagnosed from 5-21 except for hyperactive/impulsive adhd, which isn't diagnosed past 13-14 years

In [50]:
fig = px.scatter(df, x="Identifiers", y="Age", color="DX_01",
                hover_name="DX_01", log_x=False, size_max=60) # size="Age"
fig.update_layout(showlegend=False)
fig.update_xaxes(showticklabels=False, title='Participants')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()

## Participant count as a function of age

In [162]:


df1 = df[df['DX_01_Cat_new'].isin(['ADHD', 'No Diagnosis Given', 
                           'Anxiety Disorders', 
                           'Specific Learning Disorder with Impairment in Reading',
                           'Autism Spectrum Disorder',
                          'Depressive Disorders'])
                        ]
tmp = df1.groupby(['Age', 'DX_01_Cat_new']).count().reset_index()

fig = px.line(data_frame=tmp, x='Age', y='Identifiers', color='DX_01_Cat_new')
fig.update_yaxes(showticklabels=True, title='# of Participants')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})


In [138]:

df1 = df[df['DX_01_Cat_new'].isin(['ADHD', 'No Diagnosis Given', 
                           'Anxiety Disorders', 
                           'Specific Learning Disorder with Impairment in Reading',
                           'Autism Spectrum Disorder',
                          'Depressive Disorders'])
                        ]
tmp = df1.groupby(['Age', 'Sex']).count().reset_index()

fig = px.line(data_frame=tmp, x='Age', y='Identifiers', color='Sex')
fig.update_xaxes(showticklabels=True, title='Age')
fig.update_yaxes(showticklabels=True, title='# of Participants')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})

In [70]:
fig = px.scatter(df, x="Identifiers", y="Age", color="DX_01_Cat_new",
                hover_name="DX_01_Cat_new", log_x=False, size_max=60) # size="Age"
fig.update_layout(showlegend=False)
fig.update_xaxes(showticklabels=False, title='Participants')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()

## there are more than double the females with major depressive disorder (including persistent depressive disorder) than males

In [157]:
diagnosis_grouped = df.groupby(['DX_01_Cat_new', 'Sex']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'comorbidities': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)
diagnosis_grouped['percentage'] = round((diagnosis_grouped['count'] / diagnosis_grouped['count'].sum()) * 100).astype(int).astype(str) + '%'


fig = px.bar(diagnosis_grouped.sort_values(by='count', ascending=False).head(18), 
             x='count', y='DX_01_Cat_new', color='Sex', orientation='h', text='percentage')
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()


In [159]:
diagnosis_grouped = df.groupby(['DX_01_Cat_new']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'comorbidities': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)
diagnosis_grouped['percentage'] = round((diagnosis_grouped['count'] / diagnosis_grouped['count'].sum()) * 100).astype(int).astype(str) + '%'


fig = px.bar(diagnosis_grouped.sort_values(by='count', ascending=False).head(8), 
             x='count', y='DX_01_Cat_new', orientation='h', text='percentage')
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()


## mood and anxiety disorders are only disorders where females outnumber males

In [78]:
diagnosis_grouped = df.groupby(['DX_01', 'Sex']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'comorbidities': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)


fig = px.bar(diagnosis_grouped.groupby(['DX_01', 'Sex']).sum().reset_index().sort_values(by='count', ascending=False), 
             x='count', y='DX_01', color='Sex', orientation='h', text='count')
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()


## disorders with n>150: neurodevelopmental disorders (including autism, adhd, and neurocognitive/intellectual), anxiety, depression, control group (no diagnosis given)

In [79]:
diagnosis_grouped = df.groupby(['DX_01', 'DX_01_Cat_new', 'Sex']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'comorbidities': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)


fig = px.bar(diagnosis_grouped.groupby(['DX_01_Cat_new']).sum().reset_index().sort_values(by='count', ascending=False), 
             x='count', y='DX_01_Cat_new',  orientation='h', text='count')
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()


## 59% of participants have a comorbidity

In [58]:
fig = px.histogram(df, x="comorbidities", nbins=10, color="Sex")
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()

## psychosis disorders have most comorbidities (and females > males)

In [59]:
diagnosis_grouped = df.groupby(['DX_01', 'DX_01_Cat', 'Sex']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'comorbidities': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped[diagnosis_grouped['comorbidities']>0]
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)


fig = px.bar(diagnosis_grouped.sort_values(by='comorbidities', ascending=False).head(20), 
             x='comorbidities', y='DX_01', color='Sex', orientation='h', barmode="group")
fig.update_xaxes(range=[2,6])
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()


## Majority of sample have a general functioning score between 60-70%

In [60]:
fig = px.histogram(df_CGAS, x="CGAS_Score",  nbins=10)
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()

## Across broad categories, children on the schizophrenia spectrum have the lowest general functioning

In [61]:
diagnosis_grouped = df_CGAS.groupby(['DX_01_Cat']).agg({
                'Age': 'mean', 'Identifiers': 'count', 'CGAS_Score': 'mean'
                }).reset_index()
diagnosis_grouped = diagnosis_grouped.rename({'Identifiers': 'count', 'Age': 'mean_age'}, axis=1)


fig = px.bar(diagnosis_grouped, 
             x='CGAS_Score', y='DX_01_Cat', orientation='h')
fig.update_xaxes(range=[35,80])
fig.update_yaxes(title='')
fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                  'paper_bgcolor': 'rgba(0,0,0,0)'})
fig.show()